# DermAlign-Net on Kaggle / Colab

Runs the full pipeline. Turn on a GPU (T4 / P100 or better) and **Internet** (Kaggle: Settings > Internet on).

Option A (default): `download_data.py` downloads everything into the project's `data/` folder.
Option B: attach HAM10000 / ISIC 2018 test datasets on Kaggle and run `prepare_data.py` with their paths (second code cell).

Kaggle sessions stop after ~12 h: finished runs are skipped, so save `runs/` as an output dataset and re-attach it to continue.

In [ ]:
import os
REPO = '/kaggle/working/dermalign'      # where the unzipped project lives (Colab: '/content/dermalign')
WORK = REPO
os.chdir(REPO)
!pip -q install 'timm>=1.0'
# Option A: download everything into ./data and build ./data/master.csv
!python download_data.py --with_test_meta

In [ ]:
# Option B (only if you attached the datasets instead of downloading): edit paths and run
# !python prepare_data.py --ham_dir /kaggle/input/skin-cancer-mnist-ham10000 \
#     --ham_masks_dir /kaggle/input/ham10000-lesion-segmentations \
#     --isic18_test_dir /kaggle/input/<isic2018-test>/ISIC2018_Task3_Test_Input \
#     --cache_dir data/cache512 --dhash --check_test_overlap --out data/master.csv

## Stages
Run one stage per session if needed. `infl` and `abl` use 224 px; `base` and `final` use 384 px.

In [ ]:
ENV = f"CSV={WORK}/data/master.csv RUNS={WORK}/runs OUT={WORK}/results WORKERS=4 BS=32"
!{ENV} STAGES="infl" bash run_all.sh

In [ ]:
!{ENV} STAGES="abl" bash run_all.sh

In [ ]:
!{ENV} STAGES="base" bash run_all.sh

In [ ]:
!{ENV} STAGES="final" bash run_all.sh

In [ ]:
!{ENV} STAGES="xai eval" bash run_all.sh

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{WORK}/results/report.md').read()))
import glob
for f in sorted(glob.glob(f'{WORK}/results/fig_*.png'))[:6]:
    display(Image(f))